In [0]:
%sql
-- Fetch the qdp_account_transactions document data product created by 01. Quantexa Processing

-- 01. Create variables for use in this notebook
-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;



DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;




DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;

-- 2. Remove all Account Transaction records for the Tennant


DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
  
;
DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_currency s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_risk_score s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_gold.qdp_banking_supply_chain.sat_warning_signal s
WHERE s.banking_supply_chain_id IN (
  SELECT DISTINCT banking_supply_chain_id
  FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain
  WHERE tennant_id = :p_tennant_id)
;


-- 3. Select the source data from qdp_aggregated_transaction in Silver
CREATE OR REPLACE TEMP VIEW temp_transactions AS
SELECT 
    h.tennant_id,
    t.tennant_name,
    t.network_name,
    h.bene_account_entity_id,
    h.bene_account_id,
    h.orig_account_entity_id,
    h.orig_account_id,
    s.amount,
    s.datetime,
    s.description,
    s.beneficiary_name,
    s.original_account_name,
    s.period_start,
    s.period_end
  FROM demo_banking_silver.qdp_transactions_all.hub_transaction h
  JOIN demo_banking_silver.qdp_transactions_all.sat_transaction s
    ON h.transaction_id = s.transaction_id
  JOIN demo_banking_silver.qdp_refcodes_all.tennant t
   ON h.tennant_id = t.tennant_id
  WHERE h.tennant_id = 999
  ;


SELECT * FROM temp_transactions;



-- 4. Create a Summary View ready for posting into qdp_baking_supply_chain in Gold

CREATE OR REPLACE TEMP VIEW summary_transactions AS
SELECT
    tennant_id,
    tennant_name,
    network_name,
    bene_account_id,
    bene_account_entity_id,
    orig_account_id,
    orig_account_entity_id,
    count(*) AS total_transactions,
    sum(amount) AS total_value,
    min(amount) AS min_value,
    max(amount) AS max_value,
    avg(amount) AS avg_value,
    min(datetime) AS first_transaction_date,
    max(datetime) AS last_transaction_date

  FROM temp_transactions
--  GROUP BY bene_account_entity_id, orig_account_entity_id
  GROUP BY tennant_id, tennant_name, network_name, bene_account_id, bene_account_entity_id, orig_account_id, orig_account_entity_id
;

SELECT * FROM summary_transactions;




-- 5. Create the  hub_individual table records
INSERT INTO demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain (
    tennant_id,
    tennant_name,
    network_name,
    bene_account_id,
    bene_account_entity_id,
    orig_account_id,
    orig_account_entity_id
    )
SELECT 
    st.tennant_id,
    st.tennant_name,
    st.network_name,
    st.bene_account_id, 
    st.bene_account_entity_id, 
    st.orig_account_id, 
    st.orig_account_entity_id 
  FROM summary_transactions st
;

SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain WHERE tennant_id = :p_tennant_id;

  
-- 6. Create the  sat_banking_supply_chain table records
INSERT INTO demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain
    ( 
    tennant_id,
    tennant_name,
    network_name,
    banking_supply_chain_id,
    bene_account_id,
    orig_account_id,
    load_datetime,
    record_source_id,
	  total_transactions,
	  total_value,
	  max_value,
	  min_value,
    avg_value,
    first_transaction_date,
		last_transaction_date
    )
SELECT
  h.tennant_id AS tennant_id,
  h.tennant_name AS tennant_name,
  h.network_name AS network_name,
  h.banking_supply_chain_id AS banking_supply_chain_id,
  h.bene_account_id AS bene_account_id,
  h.orig_account_id AS orig_account_id,
  run_timestamp AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  st.total_transactions,
  st.total_value,
  st.max_value,
  st.min_value,
  st.avg_value,
  st.first_transaction_date,
  st.last_transaction_date
FROM summary_transactions st
JOIN demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain h
  ON ((h.bene_account_id = st.bene_account_id) AND (h.orig_account_id = st.orig_account_id));
  
SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain WHERE record_source_id = :p_tennant_id;

 

-- 6. Create the money supply chain to 3 levels of originators and beneficiaries ()

-- Determine ultimate beneficiaries
CREATE OR REPLACE TEMP VIEW view_third_level_beneficiary AS
SELECT tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name, SUM(total_value) as total_value from 
demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
WHERE beneficiary_account_name NOT IN (
  SELECT originator_account_name
  FROM demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
)
AND tennant_id = 999
GROUP BY tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name
;

SELECT * FROM view_third_level_beneficiary WHERE tennant_id = :p_tennant_id;



-- Determine beneficiaries who are originators for ultimate beneficiaries
CREATE OR REPLACE TEMP VIEW view_second_level_beneficiary AS
SELECT tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name, SUM(total_value) as total_value from 
demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
WHERE beneficiary_account_name IN (
  SELECT originator_account_name
  FROM view_third_level_beneficiary
)
GROUP BY tennant_id, tennant_name, network_name, originator_account_name, beneficiary_account_name
;

SELECT * FROM view_second_level_beneficiary WHERE tennant_id = :p_tennant_id;



DELETE FROM demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 WHERE tennant_id = :p_tennant_id;

INSERT INTO demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 (
  tennant_id, 
  tennant_name,
  network_name,
  stage1, 
  stage2, 
  stage3, 
  value
)
SELECT 
  tennant_id, 
  tennant_name,
  network_name,
  NULL, 
  originator_account_name, 
  beneficiary_account_name, 
  total_value
 FROM view_third_level_beneficiary WHERE tennant_id = :p_tennant_id
;


INSERT INTO demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 (
  tennant_id, 
  tennant_name,
  network_name,
  stage1, 
  stage2, 
  stage3, 
  value
)
SELECT 
  tennant_id, 
  tennant_name,
  network_name,
  originator_account_name, 
  beneficiary_account_name, 
  NULL, 
  total_value
 FROM view_second_level_beneficiary  WHERE tennant_id = :p_tennant_id
;


SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3  WHERE tennant_id = :p_tennant_id;

  
SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain WHERE record_source_id = :p_tennant_id;



In [0]:
%sql
SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
SELECT DISTINCT bene_account_id FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain WHERE record_source_id = :p_tennant_id;

In [0]:
%sql
SELECT DISTINCT orig_account_id FROM demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain WHERE record_source_id = :p_tennant_id;

In [0]:
%sql
SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains;

In [0]:
%sql
SELECT * FROM demo_banking_silver.qdp_accounts_all.view_organisation_accounts;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_12166451'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql

DELETE FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE account_id = 353;

INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  353,
  '12166451',
  'POST-IT OUT LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;


In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_43326478'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  354,
  '43326478',
  'CARDS & CO LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;


In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_76981356'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  355,
  '76981356',
  'CARDS & CO LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_54772848'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  356,
  '54772848',
  'PAPERCLIPZ',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_63778378'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  357,
  '63778378',
  'PAPERCLIPZ',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_1373521'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  357,
  '63778378',
  'PAPERCLIPZ',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_1373521'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  358,
  '1373521',
  'CARDBOARD COLLECTIONS LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_63244861'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  359,
  '63244861',
  'CARDBOARD COLLECTIONS LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_46235895'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  360,
  '46235895',
  'PRINTERS & INK LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_13267312'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  361,
  '13267312',
  'PRINTERS & INK LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account (
  tennant_id,
  account_entity_id) VALUES (
    999,
    'ACCOUNT_ENTITY_39458374'
  )
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 999;

In [0]:
%sql
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account (
  account_id, 
  account_number,
  account_name,
  data_source_code,
  type_code,
  status_code,
  country_code,
  is_bank_account,
  load_datetime,
  record_source_id

)
VALUES (
  362,
  '39458374',
  'PRINTERS & INK LTD',
  'CUSTOMER ACCOUNT',
  'Business Current Account',
  'Active',
  'GBR',
  true,
  current_timestamp(),
  999
);
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;

In [0]:
%sql
SELECT s.account_id
  FROM demo_banking_silver.qdp_accounts_all.hub_account h,
  demo_banking_silver.qdp_accounts_all.sat_account s
  WHERE h.account_id = s.account_id
  AND h.tennant_id = 999
  ORDER BY h.account_id;

In [0]:
%sql
SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains;

In [0]:
%sql
SELECT
  h.tennant_id as tennant_id,
  h.tennant_name AS tennant_name,
  h.network_name AS network_name,
--  beneacc.account_name as beneficiary_account_name,
--  beneacc.account_number as beneficiary_account_number,
--  cast(beneacc.is_customer_acccount as STRING) as is_beneficiary_customer,
--  origacc.account_name as originator_account_name,
--  origacc.account_number as originator_account_number,
--  cast(origacc.is_customer_acccount as STRING) as is_originator_customer,
  s.total_value,
  s.max_value,
  s.min_value,
  s.avg_value,
  s.first_transaction_date,
  s.last_transaction_date,
--  beneacc.country_code as beneficiary_account_country,
--  origacc.country_code as originator_account_country,
  h.banking_supply_chain_id,
  h.bene_account_id,
  h.bene_account_entity_id,
  h.orig_account_id,
  h.orig_account_entity_id,
  h.period_start as hub_period_start,
  h.period_end as hub_period_end,
  s.period_start,
  s.period_end,
  s.load_datetime,
  s.record_source_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id
FROM
  demo_banking_gold.qdp_banking_supply_chain.hub_banking_supply_chain h
    JOIN demo_banking_gold.qdp_banking_supply_chain.sat_banking_supply_chain s
      ON h.banking_supply_chain_id = s.banking_supply_chain_id
--    JOIN demo_banking_silver.qdp_account_all.sat_account beneacc
--      ON beneacc.account_id = h.bene_account_id
--    JOIN demo_banking_silver.qdp_accounts_all.sat_account origacc
--      ON origacc.account_id = h.orig_account_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      ON h.tennant_id = t.tennant_id
  WHERE h.tennant_id = 999    
--ORDER BY
--  beneacc.account_name